# Ensemble Learning

Coursework study extract, week 6.

Compare a single tree with bagging, random forests, extra trees, and boosting; inspect OOB scores and impurity importance. The exercise selects the best model using the displayed test scores, so the final error analysis is exploratory and selection-biased. Use training-only CV plus a new holdout before making generalization claims.

Adapted from completed code in `MCAI500_M6_In_Class_Activity.ipynb`. Instructor-provided scaffolding and pair-work context are acknowledged; authorship of individual lines cannot be independently separated. See [ATTRIBUTION.md](../ATTRIBUTION.md). Teaching prompts, supplied solutions, saved outputs, and identifying metadata are omitted.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, ExtraTreesClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
RANDOM_STATE = 42
sns.set_theme(style="whitegrid")


In [ ]:
cancer = load_breast_cancer(as_frame=True)
X = cancer.data
y = cancer.target

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Target classes:", dict(enumerate(cancer.target_names)))


In [ ]:
baseline_tree = DecisionTreeClassifier(
    max_depth=4,
    random_state=RANDOM_STATE
)

baseline_tree.fit(X_train, y_train)
baseline_pred = baseline_tree.predict(X_test)

print(f"Test accuracy: {accuracy_score(y_test, baseline_pred):.2%}")
print("Tree depth:", baseline_tree.get_depth())
print("Number of leaves:", baseline_tree.get_n_leaves())


In [ ]:
bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=RANDOM_STATE),
    n_estimators=200,
    bootstrap=True,
    oob_score=True,
    random_state=RANDOM_STATE,
    n_jobs=1
)

forest = RandomForestClassifier(
    n_estimators=200,
    max_features="sqrt",
    oob_score=True,
    random_state=RANDOM_STATE,
    n_jobs=1
)

extra_trees = ExtraTreesClassifier(
    n_estimators=200,
    random_state=RANDOM_STATE,
    n_jobs=1
)

ensemble_models = {
    "Bagging": bagging,
    "Random Forest": forest,
    "Extra Trees": extra_trees
}

rows = []

for name, model in ensemble_models.items():
    model.fit(X_train, y_train)

    rows.append({
        "Model": name,
        "Test accuracy": accuracy_score(
            y_test, model.predict(X_test)
        ),
        "OOB accuracy": getattr(model, "oob_score_", np.nan)
    })

comparison = pd.DataFrame(rows)
display(comparison.round(4))


In [ ]:
baseline_result = pd.DataFrame([{
    "Model": "Single tree",
    "Test accuracy": accuracy_score(y_test, baseline_pred)
}])

plot_results = pd.concat(
    [baseline_result, comparison[["Model", "Test accuracy"]]],
    ignore_index=True
)

ax = plot_results.plot.bar(
    x="Model",
    y="Test accuracy",
    figsize=(8, 5),
    color=["gray", "steelblue", "teal", "darkorange"],
    legend=False
)

ax.set_title("Single Tree vs Tree Ensembles")
ax.set_xlabel("")
ax.set_ylabel("Test accuracy")
ax.set_ylim(0, 1)
ax.bar_label(ax.containers[0], fmt="%.3f", padding=3)

plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
importance = pd.Series(
    forest.feature_importances_,
    index=X_train.columns,
    name="Importance"
).sort_values(ascending=False)

display(importance.head(10).to_frame().round(4))

importance.head(10).sort_values().plot.barh(
    figsize=(9, 5),
    color="teal"
)

plt.xlabel("Feature importance — mean decrease in impurity")
plt.title("Top 10 Random Forest Feature Importances")
plt.tight_layout()
plt.show()


In [ ]:
ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(
        max_depth=1,
        random_state=RANDOM_STATE
    ),
    n_estimators=100,
    learning_rate=0.5,
    random_state=RANDOM_STATE
)

gradient = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=2,
    random_state=RANDOM_STATE
)

boosting_models = {
    "AdaBoost": ada,
    "Gradient Boosting": gradient
}

rows = []

for name, model in boosting_models.items():
    model.fit(X_train, y_train)

    rows.append({
        "Model": name,
        "Test accuracy": accuracy_score(
            y_test, model.predict(X_test)
        )
    })

best_ensemble_row = comparison.loc[
    comparison["Test accuracy"].idxmax(),
    ["Model", "Test accuracy"]
].to_dict()

rows.append(best_ensemble_row)

boosting_results = pd.DataFrame(rows).sort_values(
    "Test accuracy",
    ascending=False
)

display(boosting_results.round(4))


In [ ]:
all_models = {
    "Single tree": baseline_tree,
    **ensemble_models,
    **boosting_models
}

final_results = pd.DataFrame([
    {
        "Model": name,
        "Test accuracy": accuracy_score(
            y_test, model.predict(X_test)
        )
    }
    for name, model in all_models.items()
]).sort_values("Test accuracy", ascending=False)

display(final_results.round(4))

best_name = final_results.iloc[0]["Model"]
best_model = all_models[best_name]
best_pred = best_model.predict(X_test)

print("Selected model:", best_name)

cm = confusion_matrix(y_test, best_pred, labels=[0, 1])

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    xticklabels=cancer.target_names,
    yticklabels=cancer.target_names
)

plt.xlabel("Predicted class")
plt.ylabel("Actual class")
plt.title(f"{best_name} — Confusion Matrix")
plt.show()

print(classification_report(
    y_test,
    best_pred,
    labels=[0, 1],
    target_names=cancer.target_names,
    digits=3
))


## Review notes

Compare a single tree with bagging, random forests, extra trees, and boosting; inspect OOB scores and impurity importance. The exercise selects the best model using the displayed test scores, so the final error analysis is exploratory and selection-biased. Use training-only CV plus a new holdout before making generalization claims.

These are learning experiments, not production systems. Scores depend on the data split and package versions. Generated commentary in this publication is editorial context added during preparation, not a claim that it was part of the original class submission.